# Final patent submission notebook

Selected on held-out anchors only: **raw_code**, validation Pearson **0.819542**.

Attach the official competition input and the pretrained output from the accompanying weight helper. Internet must be off; enable a GPU. This notebook trains the selected model and writes submission.csv. Publishing and competition submission were approved by the student on 12 November 2026.


## 1. Setup

In [1]:
import glob
import os
# Kaggle offers two T4s; use one to preserve the measured effective batch size of 32.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)
from pathlib import Path


torch 2.11.0+cu130 | transformers 5.20.2 | device: cuda


## 2. Configuration

These are the settings you will change in your experiments. Change **one** setting at a time so you can tell what caused a difference.

In [2]:
MODEL_ID = "microsoft/deberta-v3-small"
MAX_LEN, EPOCHS, LR, BATCH_SIZE = 64, 3, 2e-5, 32
VAL_FRACTION = 0.2
OUT_DIR = os.environ.get("PATENT_OUTPUT_DIR") or ("/kaggle/working" if Path("/kaggle/working").is_dir() else "patent_final_outputs")
Path(OUT_DIR).mkdir(parents=True,exist_ok=True)
FINAL_CONTEXT_MODE = 'raw_code'  # Chosen using validation only.
assert FINAL_CONTEXT_MODE in {"none", "raw_code", "section_title"}, "Set the measured validation-selected context mode."

def find_one(pattern):
    hits=sorted(glob.glob(f"/kaggle/input/**/{pattern}",recursive=True))
    if len(hits)>1:
        raise ValueError(f"Ambiguous attached inputs for {pattern}; use an explicit environment path.")
    return os.path.dirname(hits[0]) if hits else None

DATA_DIR=os.environ.get("PATENT_DATA_DIR") or find_one("us-patent-phrase-to-phrase-matching/train.csv")
assert DATA_DIR, "Attach the official competition directory or provide explicit PATENT_DATA_DIR."
MODEL_PATH=os.environ.get("PATENT_MODEL_PATH") or find_one(f"{MODEL_ID.split('/')[-1]}/config.json")
assert MODEL_PATH and Path(MODEL_PATH,"config.json").is_file(), "Attach the saved pretrained checkpoint for offline use."
print("data:",DATA_DIR,"| local checkpoint:",MODEL_PATH)


data: /kaggle/input/competitions/us-patent-phrase-to-phrase-matching | local checkpoint: /kaggle/input/notebooks/vanshjoshi73/lab-7-deberta-offline-weights-vansh-joshi/deberta-v3-small


## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
assert {"id","anchor","target","context","score"} <= set(train.columns)
assert {"id","anchor","target","context"} <= set(test.columns)
assert "score" not in test.columns, "Do not use released hidden-test labels."
assert len(train)==36473, "Use the official 36,473-row competition training set only."
assert train.score.isin([0,.25,.5,.75,1]).all()
assert train.id.is_unique and test.id.is_unique
# The official 36-row visible test file is a format sample whose IDs occur in training.
# Never score on it or look up hidden-test labels; validation holds out whole anchors.
print("Current test IDs also present in training:", len(set(train.id) & set(test.id)))
assert train[["id","anchor","target","context","score"]].notna().all().all()
assert test[["id","anchor","target","context"]].notna().all().all()
print("train:",train.shape,"| current test rows:",len(test))
print(train.score.value_counts().sort_index())
train.head()


Current test IDs also present in training: 36
train: (36473, 5) | current test rows: 36
score
0.00     7471
0.25    11519
0.50    12300
0.75     4029
1.00     1154
Name: count, dtype: int64


,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00


## 4. Model input

The anchor is the first segment and the target is the second. The context representation is **raw_code**, selected from the three controlled runs. Section-title mode appends the official starter's short CPC section description; raw-code mode appends the original code; no-context mode appends nothing. Joint attention makes this a cross-encoder.


In [4]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def add_inputs(df):
    df = df.copy()
    df["text_a"] = df["anchor"]
    suffix={"none":pd.Series("",index=df.index),"raw_code":"; "+df.context,
            "section_title":"; "+df.context.str[0].map(CPC_SECTIONS)}[FINAL_CONTEXT_MODE]
    df["text_b"] = df["target"] + suffix
    return df


train, test = add_inputs(train), add_inputs(test)
train[["text_a", "text_b", "score"]].head()

,text_a,text_b,score
0,abatement,abatement of pollution; A47,0.50
1,abatement,act of abating; A47,0.75
2,abatement,active catalyst; A47,0.25
3,abatement,eliminating process; A47,0.50
4,abatement,forest region; A47,0.00


## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [5]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

train rows 28914 | val rows 7559 | val anchors 147


## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

[transformers] The tokenizer you are loading from '/kaggle/input/notebooks/vanshjoshi73/lab-7-deberta-offline-weights-vansh-joshi/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

[CLS] abatement[SEP] abatement of pollution; A47[SEP]


## 7. Fine-tuning

With num_labels=1 and floating labels, the Transformer uses a regression head and mean squared error. The competition metric is Pearson correlation between predictions and true scores. Pearson measures linear association: it is unchanged by adding a constant or multiplying predictions by a positive constant, so it does not assess absolute calibration. Multiplying by a negative constant reverses its sign. It is not merely a ranking metric: a nonlinear monotone transformation can preserve every rank but change Pearson (Spearman measures rank correlation).


In [7]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}


# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, torch_dtype=torch.float32, local_files_only=True)
# Transformers 5.20 removed warmup_ratio; 272 steps is the same 10%
# warmup for ceil(28,914 / 32) * 3 = 2,712 optimizer steps on one GPU.
import math
warmup_steps = math.ceil(0.1 * math.ceil(len(ds_trn) / BATCH_SIZE) * EPOCHS)
print("Warmup optimizer steps:", warmup_steps)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/vanshjoshi73/lab-7-deberta-offline-weights-vansh-joshi/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING   

Warmup optimizer steps: 272


Epoch,Training Loss,Validation Loss,Pearson
1,0.029074,0.026800,0.804930
2,0.022153,0.024066,0.815118
3,0.018638,0.024354,0.819209


TrainOutput(global_step=2712, training_loss=0.03008100973403735, metrics={'train_runtime': 213.3352, 'train_samples_per_second': 406.6, 'train_steps_per_second': 12.712, 'total_flos': 324744429488520.0, 'train_loss': 0.03008100973403735, 'epoch': 3.0})

## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [8]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

Validation Pearson: 0.8192


## 9. Submission

The competition expects /kaggle/working/submission.csv with columns id,score and one row per current test row. The hidden test set can have a different size from the visible sample. Clipping scores to [0,1] enforces the target range; this nonlinear operation can change Pearson. No claim about its metric effect is made without measuring it.


In [9]:
test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

,id,score
0,4112d61851461f60,0.562012
1,09e418c93a776564,0.809570
2,36baf228038e314b,0.544922
3,1f37ead645e7f0c8,0.347168
4,71a5b6ad068d531f,0.000000


## Submission status

Tasks 1-4 use the measured local experiments. The validation-selected raw-CPC configuration also completed an offline Kaggle GPU run and a scored late submission on 12 November 2026. Local validation r = 0.8195; the Kaggle execution gives r = 0.8192. The final notebook is public, and the separate Submissions screenshot records both hidden-test scores.

One scored competition submission: version 2, script ID 368715520, public 0.7904, private 0.8102. No test score influenced model selection. The first saved Kaggle execution failed before training because Transformers 5.20 removed warmup_ratio; it was not submitted. The successful version uses the equivalent 272 warmup steps (10% of 2,712), one T4, and unchanged model, split, batch size and three-epoch budget.

https://www.kaggle.com/code/vanshjoshi73/lab-7-patent-phrase-matching-vansh-joshi?scriptVersionId=368715520


## Revision provenance

Prepared with Codex assistance from the official course notebook. No other student solution was adapted. These saved outputs are the actual Kaggle execution, exported on 12 November 2026. The separate experiment notebook records the local runs. The student approved publication and submission. Only this provenance/status Markdown was updated after export; code and execution outputs are unchanged.
